In [1]:
import torch
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
%load_ext autotime

# ── Config ─────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
DEVICE   = "cuda"
DTYPE    = torch.float16   # use float16 if bfloat16 unsupported on your GPU


# ── Load model (do once, reuse for many meshes) ────────────────────────────
def load_model(model_id: str = MODEL_ID):
    print(f"Loading {model_id} ...")
    processor = AutoProcessor.from_pretrained(model_id)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",      # auto-assigns layers across GPU (and CPU if needed)
    )
    model.eval()
    print("Model ready.")
    return model, processor




Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


time: 166 μs (started: 2026-05-06 14:09:12 +07:00)


In [2]:
def load_and_resize(img_path, image_size):
    img = Image.open(img_path).convert("RGB")
    img.thumbnail(image_size)
    return img

def run_multimodal_prompt(
    model,
    processor,
    prompt,
    image_paths=None,
    max_new_tokens=256,
    image_size=(256,512)
):
    content = []

    # Add images
    if image_paths:
        for path in image_paths:
            img = load_and_resize(path, image_size)
            content.append({"type": "image", "image": img})

    # Add text
    content.append({"type": "text", "text": prompt})

    messages = [
        {
            "role": "user",
            "content": content
        }
    ]

    # Build input
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[c["image"] for c in content if c["type"] == "image"] or None,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,  
        )

    # Decode
    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]

    output_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return output_text


time: 593 μs (started: 2026-05-06 14:09:16 +07:00)


In [3]:
model, processor = load_model()

Loading Qwen/Qwen2.5-VL-7B-Instruct ...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model ready.
time: 3min 54s (started: 2026-05-06 14:09:18 +07:00)


In [17]:
out = run_multimodal_prompt(
    model,
    processor,
    "give me 6 - 6length random numbers",max_new_tokens=516*2
)

print(out)

Here are six randomly generated 6-digit numbers:

1. 456789
2. 134567
3. 890123
4. 246810
5. 735986
6. 317452
time: 1.65 s (started: 2026-05-06 14:19:19 +07:00)


In [5]:
out = run_multimodal_prompt(
    model,
    processor,
    "what is the color of the kitten eyes ?",
    image_paths=["silver-tabby-cat-sitting-on-green-background-free-photo.jpg"],max_new_tokens=516,image_size =(512,512)
)

print(out)

The kitten in the picture has green eyes.
time: 2.47 s (started: 2026-05-06 14:14:08 +07:00)


In [9]:
out = run_multimodal_prompt(
    model,
    processor,
    """These images show multiple views of the same fractured 3D object. 
The orange/red-highlighted regions indicate the fracture surface — 
the face where the a smaller piece have broke off from the object.

Please describe:
1. What the overall object appears to be (shape, category, likely material)
2. The approximate size and position of the fracture surface relative to 
   the whole object (like 30%)
3. The fracture geometry — is it flat, jagged, curved, or irregular?
4. Which part of the missing piece of this object likely came from 
   (e.g. "a corner piece", "a side slab", "a central chunk")
5. The orientation of the break — which direction did the object split 
   (e.g. "broke horizontally", "diagonal shear", "split along a vertical plane")

Be specific about what you can see across all views combined. combine all of that into 1 line caption""",
    image_paths=["out/everyday/Vase/6af347c8e74b5b715d878ba9ec3c0d6a/fractured_25/piece_0_frac_view_frac02.png","out/everyday/Vase/6af347c8e74b5b715d878ba9ec3c0d6a/fractured_25/piece_0_frac_view_frac04.png"],max_new_tokens=516,image_size =(516,516)
)

print(out)

The 3D object appears to be a stylized flower pot or vase with a wide base tapering towards a narrower neck, made of ceramic or clay. The fracture surface is located around 30% from the top, indicating the object likely broke from a central chunk of the body, with the missing piece resembling a section of the main body rather than a corner or side slab. The fracture geometry is jagged, suggesting either a diagonal shear or split along a vertical plane, though the specific direction cannot be definitively determined without additional context.
time: 2.57 s (started: 2026-04-24 12:55:11 +07:00)


In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1e9, "GB")
print(torch.cuda.memory_reserved() / 1e9, "GB")

True
16.593924608 GB
16.6723584 GB
time: 676 μs (started: 2026-05-06 14:16:02 +07:00)
